In [1]:
import gzip
import time
import tracemalloc
from collections import defaultdict
from fileReader import stream_dataset


class TrieNode:
    def __init__(self):
        self.children = {}
        self.count = 0
        self.is_leaf = False

In [2]:
import math


class AprioriTrie:
    def __init__(self, min_support_ratio):
        self.min_support_ratio = min_support_ratio
        self.min_support_count = 0
        self.root = TrieNode()
        self.total_transactions = 0

    def _insert_candidate(self, itemset):
        node = self.root
        for item in itemset:
            if item not in node.children:
                node.children[item] = TrieNode()
            node = node.children[item]
        node.is_leaf = True

    def _exists_in_trie(self, itemset):
        node = self.root
        for item in itemset:
            if item not in node.children:
                return False
            node = node.children[item]
        return node.is_leaf

    def _count_transaction_recursive(self, node, transaction, start_idx, target_depth, current_depth):
        """Traverses Trie paths present in a single transaction down to target_depth."""
        if current_depth == target_depth:
            node.count += 1
            return

        # Pruning: remaining items in transaction must be enough to reach target_depth
        remaining_needed = target_depth - current_depth
        limit = len(transaction) - remaining_needed + 1

        for i in range(start_idx, limit):
            item = transaction[i]
            if item in node.children:
                self._count_transaction_recursive(
                    node.children[item], transaction, i + 1, target_depth, current_depth + 1
                )

    def _generate_candidates_and_prune(self, node, current_prefix, current_depth, target_parent_depth):

        if current_depth == target_parent_depth:
            items = sorted(node.children.keys())
            n = len(items)
            new_candidates = 0
            for i in range(n):
                for j in range(i + 1, n):
                    # Join step: share (k-2) prefix, differ at last item
                    candidate = current_prefix + [items[i], items[j]]

                    # Prune step: check all (k-1) subsets exist in the Trie
                    all_subsets_frequent = True
                    for skip_idx in range(len(candidate) - 2):
                        subset = candidate[:skip_idx] + candidate[skip_idx + 1 :]
                        if not self._exists_in_trie(subset):
                            all_subsets_frequent = False
                            break

                    if all_subsets_frequent:
                        child_node = node.children[items[i]]
                        child_node.children[items[j]] = TrieNode()
                        child_node.children[items[j]].is_leaf = True
                        new_candidates += 1
            return new_candidates

        total_added = 0
        for item, child in node.children.items():
            total_added += self._generate_candidates_and_prune(
                child, current_prefix + [item], current_depth + 1, target_parent_depth
            )
        return total_added

    def _prune_infrequent_nodes(self, node, current_depth, target_depth):
        """Removes nodes at target_depth whose support < min_support_count."""
        if current_depth == target_depth - 1:
            to_delete = []
            for item, child in node.children.items():
                if child.count < self.min_support_count:
                    to_delete.append(item)
                    
            for item in to_delete:
                del node.children[item]
            return len(node.children)

        frequent_count = 0
        for item, child in list(node.children.items()):
            frequent_count += self._prune_infrequent_nodes(child, current_depth + 1, target_depth)
        return frequent_count

    def fit(self, filepath):
        self.root = TrieNode()

        item_counts = defaultdict(int)
        tx_count = 0
        for tx in stream_dataset(filepath):
            tx_count += 1
            for item in tx:
                item_counts[item] += 1

        self.total_transactions = tx_count
        self.min_support_count = math.ceil(self.min_support_ratio * self.total_transactions)

        f1_count = 0
        for item in sorted(item_counts.keys()):
            if item_counts[item] >= self.min_support_count:
                node = TrieNode()
                node.count = item_counts[item]
                node.is_leaf = True
                self.root.children[item] = node
                f1_count += 1

        total_frequent_itemsets = f1_count
        k = 2

        # --- LEVEL k >= 2: Generate C_k -> 1 Dataset Scan -> Prune to F_k ---
        while True:
            # 1. Generate C_k from F_{k-1} inside the Trie
            candidates_generated = self._generate_candidates_and_prune(self.root, [], 0, k - 2)
            if candidates_generated == 0:
                break

            # 2. Scan the dataset ONCE for level k
            for tx in stream_dataset(filepath):
                if len(tx) >= k:
                    self._count_transaction_recursive(self.root, tx, 0, k, 0)

            # 3. Prune infrequent nodes to leave only F_k
            fk_count = self._prune_infrequent_nodes(self.root, 0, k)
            if fk_count == 0:
                break

            total_frequent_itemsets += fk_count
            k += 1

        return total_frequent_itemsets, self.total_transactions, self.min_support_count


In [3]:
def run_benchmark(dataset_name, filepath, support_thresholds):
    print(f"\n{'='*100}")
    print(f"Dataset: {dataset_name}")
    print(f"{'MinSup (%)':<12} | {'Total Transactions':<15} | {'Min Support Count':<15} | {'Freq Itemsets':<15} | {'Runtime (s)':<14} | {'Peak Memory (MB)':<16}")
    print(f"{'-'*100}")

    for sup in support_thresholds:
        tracemalloc.start()

        start_time = time.perf_counter()

        model = AprioriTrie(min_support_ratio=sup)
        freq_count, total_transactions, min_support_count = model.fit(filepath)

        elapsed = time.perf_counter() - start_time
        _, peak_bytes = tracemalloc.get_traced_memory()
        
        tracemalloc.stop()

        peak_mb = peak_bytes / (1024 * 1024)
        print(f"{sup*100:<11.2f}% | {total_transactions:<18.2f} | {min_support_count:<18.1f} | {freq_count:<15} | {elapsed:<14.4f} | {peak_mb:<16.4f}")
        

In [4]:
if __name__ == "__main__":
    # Dense datasets need HIGH thresholds so combinatorial explosion doesn't stall Python
    dense_configs = [
        ("Chess --> Dense Dataset", "datasets/chess.dat.gz", [0.70, 0.75, 0.80, 0.85, 0.90]),
        ("Mushroom --> Dense Dataset", "datasets/mushroom.dat.gz", [0.35, 0.40, 0.45, 0.50, 0.60]),
    ]

    # Sparse datasets need LOW thresholds to find meaningful patterns
    sparse_configs = [
        ("Retail --> Sparse Dataset", "datasets/retail.dat.gz", [0.005, 0.01, 0.015, 0.02, 0.03]),
        ("kosarak --> Sparse Dataset", "datasets/kosarak.dat.gz", [0.005, 0.01, 0.015, 0.02, 0.025]),
    ]

    for name, path, thresholds in dense_configs + sparse_configs:
        run_benchmark(name, path, thresholds)


Dataset: Chess --> Dense Dataset
MinSup (%)   | Total Transactions | Min Support Count | Freq Itemsets   | Runtime (s)    | Peak Memory (MB)
----------------------------------------------------------------------------------------------------
70.00      % | 3196.00            | 2238.0             | 48731           | 259.9699       | 12.8614         
75.00      % | 3196.00            | 2397.0             | 20993           | 101.6049       | 5.7379          
80.00      % | 3196.00            | 2557.0             | 8227            | 39.3975        | 2.4866          
85.00      % | 3196.00            | 2717.0             | 2669            | 10.7441        | 1.0699          
90.00      % | 3196.00            | 2877.0             | 622             | 2.8409         | 0.5498          

Dataset: Mushroom --> Dense Dataset
MinSup (%)   | Total Transactions | Min Support Count | Freq Itemsets   | Runtime (s)    | Peak Memory (MB)
-------------------------------------------------------------------

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/kosarak.dat.gz'